In [67]:
# Set up environment variables
from dotenv import load_dotenv
load_dotenv()

True

In [68]:
# Rough plan
# Set up s3 KB 
# Choice: Simple llm call via langchain fetching context before OR Agent with Rag tool
# Choice: Langchain atraight to s3 vector and manual chunking OR Langchain with bedrock KB + s3vector backend: No manual chunking but needs permissions


In [70]:
# Set up s3 vector storage through langchain:

# We will use an s3 vector bucket which allows us to use the bucket as a vector db without without managing it ourselves - AWS handles it
import boto3
client = boto3.client('s3vectors',
                       region_name="eu-central-1",
                       )
try:
    response = client.create_vector_bucket(
        vectorBucketName = 'test-bucket',
    )
    print(response)
except client.exceptions.ConflictException:
    print("The bucket exists already")


# We also need to define a vector index
try:
    response = client.create_index(
        vectorBucketName='test-bucket',
        indexName='test-index',
        dataType='float32',
        dimension=1536, # This is dependent on the embedding model we will use later
        distanceMetric='cosine',
    )
except client.exceptions.ConflictException:
    print("The index exists already")



The bucket exists already
The index exists already


In [71]:
# We now want to feed chunks of our html documents into the vector bucket
# However, we are facing two problems:
# Problem 1: Our Raw html documents are noisy, we just want to extract content.

from bs4 import BeautifulSoup
# Load our example html (TUM Examinations html page)
with open ("../../html_samples/tum_examinations.html") as f:
    html_text = f.read()
print(f"The full file has {len(html_text)} characters")

# Let's extract only the main content from the file to reduce noise
def extract_main_content(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")
    
    # Try to find the main content container
    # TUM uses id="main-content" (from the skip-nav link)
    main = (
        soup.find(id="main-content")
        or soup.find("main")
        or soup.find("article")
    )
    return str(main)

html_text = extract_main_content(html_text)

# Furthermore let's remove some noisy tags all together
def strip_nav_noise(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup.find_all(["nav", "header", "footer", "aside", "noscript", "script", "style"]):
        tag.decompose()
    return str(soup)

html_text = strip_nav_noise(html_text)
# Be aware this html cleaning is specific to this site and may be a bit crude - it just serves as an example

print(f"The stripped file has {len(html_text)} characters")

The full file has 742337 characters
The stripped file has 17515 characters


In [72]:
# Problem 2: We need to chunk the content into smaller pieces to use them effectively in a vector db.
# Also the html text still includes a bunch of tags.
# Luckily langchain has inbuilt solutions for this:
from langchain_text_splitters import HTMLSectionSplitter, RecursiveCharacterTextSplitter

# First we use the section splitter to split the pages into broad sections (Once again this is specific to this html page!)
headers_to_split_on = [
    ("h1", "Header 1"),
    ("h2", "Header 2"),
]

html_splitter = HTMLSectionSplitter(headers_to_split_on)
# This creates a list of langchain documents, which may still be too large to upload as one
html_header_splits = html_splitter.split_text(html_text)

# Now we define chunk sizes and further split the documents into smaller chunks
chunk_size = 600
chunk_overlap = 150
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
)

# The splits should now be ready to upload
splits = text_splitter.split_documents(html_header_splits)

In [76]:
# Now that we have our chunks ready we can upload them into out s3 vector bucket
# Langchain has inbuilt solutions to do this so we do not need to handle it manually in boto3
from langchain_aws.embeddings import BedrockEmbeddings
from langchain_aws.vectorstores.s3_vectors import AmazonS3Vectors

# Let's define an embedding model to vectorize our chunks
bedrock_client = boto3.client(
    'bedrock-runtime',
    region_name = 'eu-central-1',
    )
embedding = BedrockEmbeddings(
    client=bedrock_client,
    region_name='eu-central-1',
    model_id = "amazon.titan-embed-text-v1", # Important: This model matches the 1024 dimensions we defined for the index
    )

vector_store = AmazonS3Vectors(
    vector_bucket_name="test-bucket",
    index_name="test-index",
    embedding=embedding,
    region_name='eu-central-1',
    client=client
)

# Check if our bucket is already populated:
response = client.list_vectors(
    vectorBucketName='test-bucket',
    indexName='test-index',
    maxResults=1,
)
if not response['vectors']:
    # If the vector storage is still empty we actually add the documents
    vector_store.add_documents(
        splits
    )
    print("Added vectors succesfully")

In [82]:
# Now that we set up out vector db we can run similarity search to retrieve information
vector_store.similarity_search_with_score(
    "Who governs examinations at TUM?"
)

[(Document(id='a91c3523dc0d42289ed360ba4ebcd88f', metadata={'Header 1': '#TITLE#'}, page_content="All examinations at TUM are governed under the General Academic and Examination Regulations for Bachelor's and Master's Programs (APSO) and the General Diploma Examination Regulations (ADPO) for discontinued diploma degree programs, in conjunction with the Study and Examination Regulations (FPSO) for the respective school. Section 1 of your FPSO stipulates whether your course of study falls under ADPO or APSO."),
  0.3147532343864441),
 (Document(id='4cae25afb9f04b478f3ec61a9bb0f22a', metadata={'Header 1': '#TITLE#'}, page_content='Nonadherence to these regulations can have serious consequences, including failing the entire course of study. \n \n \n \n \n \n \n \n \n \n \n When it comes to examinations, there are three important points of contact at TUM:    1. The  office of student affairs of each TUM school  is responsible for organizing course work examinations in the individual degree 

In [ ]:
# Now that we have a simple RAG system let's build a simple agent to use it:
